## income

In [32]:
import pandas as pd
import re
import numpy as np

In [33]:
# import data

data_income = pd.read_csv(r"D:\Portfolio\Data Engineer Intern at BTS Group (Jul - Nov 2023)\Data Quality Improvement Project\notebook_python\income\1_customer_comp_income.csv" , encoding = 'utf8')


data_income.shape

(73, 12)

In [34]:
pd.set_option('display.max_rows', None)

In [35]:
data_income.head(5)

,subject,no,id,case,detail case,flag,income,range_min,range_max,less,more,spot
0,income,1,income-Range-1,Range,character,good,10K - 20K,"10,000","20,000",NaN,NaN,NaN
1,income,2,income-Range-2,Range,character,good,15K - 20K,"15,000","20,000",NaN,NaN,NaN
2,income,3,income-Range-3,Range,character,good,20K - 35K,"20,000","35,000",NaN,NaN,NaN
3,income,4,income-Range-4,Range,character,good,20K - 40K,"20,000","40,000",NaN,NaN,NaN
4,income,5,income-Range-5,Range,character,good,40K - 60K,"40,000","60,000",NaN,NaN,NaN


In [36]:
data_income.tail(5)

,subject,no,id,case,detail case,flag,income,range_min,range_max,less,more,spot
68,income,69,income-Spot-69,Spot,other,bad,xx6xxx,NaN,NaN,NaN,NaN,NaN
69,income,70,income-Spot-70,Spot,other,bad,",200.00,00",NaN,NaN,NaN,NaN,NaN
70,income,71,income-Spot-71,Spot,other,bad,0.0002,NaN,NaN,NaN,NaN,NaN
71,income,72,income-Spot-72,Spot,other,bad,0.002,NaN,NaN,NaN,NaN,NaN
72,income,73,income-Spot-73,Range,bad range,bad,ผ20K-35K,NaN,NaN,NaN,NaN,NaN


In [37]:
# remove ,
# convert T | – - to -
# convert x k m to upper case
def standard_transform(text_input): 
    replace = {
        'to': ' - ',
        'T': ' - ',
        '|': ' - ',
        'X': 'K',
        'k' : 'K',
        'm$': 'M',
        '–': ' - ',
        '-': ' - ',
        ',': ''
    }
    
    for old, new in replace.items():
        text_input = text_input.replace(old, new)
    
    # convert K to 1000 , M to 1000000
    try :
        text_input = re.sub(r'[ก-๛]' , '' , text_input)
        if re.search(r' - ' , text_input) :
            split = text_input.split(' - ')

            min_value = split[0].strip()
            max_value = split[1].strip()

            if min_value.endswith('K'):
                result_min = int(float(min_value[:-1]) * 1000)
            elif min_value.endswith('M'):
                result_min = int(float(min_value[:-1]) * 1000000)
            else :
                result_min = re.sub(r'\.[0-9]+$', '', min_value).replace('.' , '')

            if max_value.endswith('K'):
                result_max = int(float(max_value[:-1]) * 1000)
            elif max_value.endswith('M'):
                result_max = int(float(max_value[:-1]) * 1000000)                
            else:
                result_max = re.sub(r'\.[0-9]+$', '', max_value).replace('.' , '')

            result = f'{result_min} - {result_max}'
            return result
        

        def convert_char(text_input):
            if '.' in text_input.group():
                return str(int(float(text_input.group())))
            elif 'K' in text_input.group():
                return str(int(float(text_input.group()[:-1]) * 1000))
            elif 'M' in text_input.group():
                return str(int(float(text_input.group()[:-1]) * 1000000))
            else:
                return text_input.group()
            
        result = re.sub(r'(\d+(\.\d+)?K?M?)', convert_char, text_input).lower().replace('.' , '')
        return result

    except :
        return None
    




# transform data (SI unit)
data_income['income_transform'] = data_income['income'].apply(lambda income : standard_transform(income))

In [38]:
# switch position of some data
def switch_range(text_input) :
    try :
        split = text_input.split('-')
        
        min_value = int(split[0])
        max_value = int(split[1])
        
        if min_value > max_value :
            result = f"{max_value} - {min_value}"
            return result
        else :
            return text_input
    except :
        return text_input
    
    
    

# switch position of bad range
data_income['income_transform'] = data_income['income_transform'].apply(lambda income_transform : switch_range(income_transform))

In [39]:
# choose only range
def take_range(text_input):
    if text_input is None:
        return None

    try:
        if re.search(r'([0-9a-z]{1,10}\s?-\s?[0-9a-z]{1,10})', text_input):
            split = text_input.split(' - ')
            
            min_value = str(split[0])
            max_value = str(split[1])

            if min_value != max_value :
                return f"{min_value} - {max_value}"
            
    except AttributeError:
        return None
    


# create column for range
data_income['df_range'] = data_income['income_transform'].apply(lambda income_transform : take_range(income_transform))

In [40]:
# choose only less
def take_less(text_input) :
    if text_input is None:
        return None
    
    try :
        if re.search(r'^l' , text_input) :
            return text_input
    except AttributeError :
        return None
    
    


# create column for less
data_income['df_less'] = data_income['income_transform'].apply(lambda income_transform : take_less(income_transform))

In [41]:
# choose only more
def take_more(text_input) :
    if text_input is None:
        return None
    
    try :
        if re.search(r'^m' , text_input) :
            return text_input
    except AttributeError :
        return None
    
    
    

# create column for more
data_income['df_more'] = data_income['income_transform'].apply(lambda income_transform : take_more(income_transform))

In [42]:
# choose only spot
def take_spot(text_input) :
    if text_input is None:
        return None
    
    try :
        if re.search(r'([0-9]+\s?-+\s?[0-9]+)', text_input) :
            split = text_input.split(' - ')
            
            min_value = str(split[0])
            max_value = str(split[1])

            if min_value == max_value :
                return max_value
            
        elif re.search(r'(^[l,m])' , text_input) :
            return None
        elif re.search(r'([0-9a-zA-Z]{1,10}\s?-\s?[0-9a-zA-Z]{1,10})' , text_input) :
            return None
        elif re.search(r'^[0-9]+[a-zA-Z]?$' , text_input) :
            return text_input
    
        else :
            return None
    except AttributeError :
        return None
    
    
    
    
    
# create column for spot
data_income['df_spot'] = data_income['income_transform'].apply(lambda income_transform : take_spot(income_transform))

In [43]:
# create data frame for range , less , more , spot
income_range = data_income[data_income['df_range'].notnull()]
income_less = data_income[data_income['df_less'].notnull()]
income_more = data_income[data_income['df_more'].notnull()]
income_spot = data_income[data_income['df_spot'].notnull()]
income_unknown = data_income[data_income['df_range'].isnull()][data_income['df_less'].isnull()][data_income['df_more'].isnull()][data_income['df_spot'].isnull()]


# del original data frame
del data_income

C:\Users\sumse\AppData\Local\Temp\ipykernel_37396\3522385531.py:6: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  income_unknown = data_income[data_income['df_range'].isnull()][data_income['df_less'].isnull()][data_income['df_more'].isnull()][data_income['df_spot'].isnull()]
C:\Users\sumse\AppData\Local\Temp\ipykernel_37396\3522385531.py:6: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  income_unknown = data_income[data_income['df_range'].isnull()][data_income['df_less'].isnull()][data_income['df_more'].isnull()][data_income['df_spot'].isnull()]
C:\Users\sumse\AppData\Local\Temp\ipykernel_37396\3522385531.py:6: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  income_unknown = data_income[data_income['df_range'].isnull()][data_income['df_less'].isnull()][data_income['df_more'].isnull()][data_income['df_spot'].isnull()]


In [44]:
# standard format of range and spot
def char_range_spot(text_input) :
    try :
        if re.search(r'[a-zA-Zก-๛]+' , text_input) :
            return None
        elif re.search(r'[0-9]{,10}\s?-\s?[0-9]{,10}' , text_input) :
            return text_input
        elif text_input.startswith(str(0)) :
            return str(0)
        elif re.search(r'([0-9]+)' , text_input) :
            return text_input
        else:
            return None
        
    except :
        return None
    
    
    
# standard format of range and spot
income_range['df_range'] = income_range['df_range'].apply(lambda df_range : char_range_spot(df_range))
income_spot['df_spot'] = income_spot['df_spot'].apply(lambda df_spot : char_range_spot(df_spot))

In [45]:
income_spot.head(5)

,subject,no,id,case,detail case,flag,income,range_min,range_max,less,more,spot,income_transform,df_range,df_less,df_more,df_spot
37,income,38,income-Spot-38,Spot,number,good,40000,NaN,NaN,NaN,NaN,"40,000",40000,None,None,None,40000
38,income,39,income-Spot-39,Spot,character,good,50K,NaN,NaN,NaN,NaN,"50,000",50000,None,None,None,50000
39,income,40,income-Spot-40,Spot,number,good,"60,000.00",NaN,NaN,NaN,NaN,"60,000",60000,None,None,None,60000
40,income,41,income-Spot-41,Spot,number,good,"70,000",NaN,NaN,NaN,NaN,"70,000",70000,None,None,None,70000
45,income,46,income-Range-46,Range,bad range,bad,"10,001-10,001",NaN,NaN,NaN,NaN,NaN,10001 - 10001,None,None,None,10001


In [46]:
# standard format of less and more        
def char_less_more(text_input) :
    try :
        result = re.findall(r'([0-9]+)' , text_input)
        return ''.join(result)
    except :
        return None
    
    
    
    
# standard format of less and more
income_less['df_less'] = income_less['df_less'].apply(lambda df_less : char_less_more(df_less))
income_more['df_more'] = income_more['df_more'].apply(lambda df_more : char_less_more(df_more))

In [47]:
# validate consecutively number
# range
def length_range(text_input) :
    try :
        if re.search(r'([0-9]+\s?-+\s?[0-9]+)', text_input) :
            split = text_input.split(' - ')
            
            min_value = int(split[0])
            max_value = int(split[1])
            
            if 4 <= len(str(min_value)) <= 10 and 4 <= len(str(max_value)) <= 10 :
                return True
            else :
                return False
        else :
            return False
    except :
        return False
    
    
# validate length 4 <= data <= 10

# range
income_range['validate_length'] = income_range['df_range'].apply(lambda df_range : length_range(df_range))

In [48]:
# validate length 4 <= data <= 10
# all cases except range
def length(text_input) :
    try :
        text_input = re.findall(r'([0-9]+)' , text_input)
        result = text_input
        
        if 4 <= len(str(result)) <= 10 :
            return True
        else :
            return False
    except :
        return False
    


    
# less
income_less['validate_length'] = income_less['df_less'].apply(lambda df_less : length(df_less))

# more
income_more['validate_length'] = income_more['df_more'].apply(lambda df_more : length(df_more))

# spot
income_spot['validate_length'] = income_spot['df_spot'].apply(lambda df_spot : length(df_spot))

In [49]:
# segment range of data
# min_value -> min , max_value -> max
def convert_segment_range(text_input):
    try:
        split = text_input.split(' - ')
        min_value = int(split[0])
        max_value = int(split[1])

        if min_value < max_value:
            return str(min_value), str(max_value), None
        elif min_value == max_value:
            return None, None, str(max_value)
        else:
            return None, None, None
    except:
        return None, None, None
    


    
# range
income_range[['new_range_min', 'new_range_max', 'new_spot']] = income_range['df_range'].apply(lambda df_range: pd.Series(convert_segment_range(df_range)))

In [50]:
# segment less , more , spot of data
# less -> less
# more -> more
# spot -> spot
def convert_segment(text_input) :
    try :
        if re.search(r'\s' , text_input) :
            return None
        elif re.findall(r'([0-9]+)' , text_input) :
            return text_input

    except :
        return None
    
    
    
# less
income_less['new_less'] = income_less['df_less'].apply(lambda df_less : convert_segment(df_less))

# more
income_more['new_more'] = income_more['df_more'].apply(lambda df_more : convert_segment(df_more))

# spot
income_spot['new_spot'] = income_spot['df_spot'].apply(lambda df_spot : convert_segment(df_spot))

In [51]:
def income_99(text_input) :
    if text_input == None :
        return None
    elif text_input.endswith('99') :
        result = int(text_input) + 1
        return int(result)
    elif text_input.endswith('1') :
        result = int(text_input) - 1
        return int(result)
    else :
        return text_input
    



# range_min
income_range['range_min_comp'] = income_range['new_range_min'].apply(lambda new_range_min : income_99(new_range_min))

# range_max
income_range['range_max_comp'] = income_range['new_range_max'].apply(lambda new_range_max : income_99(new_range_max))

# less
income_less['less_comp'] = income_less['new_less'].apply(lambda new_less : income_99(new_less))

# more
income_more['more_comp'] = income_more['new_more'].apply(lambda new_more : income_99(new_more))

# spot
income_spot['spot_comp'] = income_spot['new_spot'].apply(lambda new_spot : income_99(new_spot))

In [52]:
# combined data frame of all cases
income = pd.concat([income_range , income_less , income_more , income_spot , income_unknown]).replace(np.nan , None)

In [53]:
selected_cols = ['id' ,'income' , 'income_transform' , 'df_range' , 'df_less' , 'df_more' , 'df_spot' , 'range_min_comp' , 'range_max_comp' , 'less_comp' , 'more_comp' , 'spot_comp' , 'validate_length']

income = income[selected_cols]

In [54]:
# validate consecutive number of range
def dup_number_range(min_value , max_value):
    try :
        digits_min = set(str(min_value))
        digits_max = set(str(max_value))
        
        if min_value == None and max_value == None :
            return False
        elif len(digits_min) == 1 and len(digits_max) == 1 :
            return False
        elif len(digits_min) >= 2 and len(digits_max) >= 2 :
            return False
        else:
            return True
    except :
        return False
    
    
    
# validate consecutive number of data

# range
income['validate_consecutive'] = income.apply(lambda income : dup_number_range('range_min_comp' , 'range_max_comp'))


In [55]:
# validate consecutive number of data
def dup_number(text_input):
    try :
        digits = set(str(text_input))
        
        if text_input == None :
            return False
        elif len(text_input) == 1 :
            return False
        elif len(digits) >= 2:
            return False
        else:
            return True
    except :
        return False
    



# less
income['validate_consecutive'] = income['less_comp'].apply(lambda new_less : dup_number(new_less))

# more
income['validate_consecutive'] = income['more_comp'].apply(lambda new_more : dup_number(new_more))

# spot
income['validate_consecutive'] = income['spot_comp'].apply(lambda new_spot : dup_number(new_spot))

In [56]:
# standard format
def combined_income(range_min, range_max, less, more, spot):
    if range_min is not None and range_max is not None:
        result_range = f'{range_min} - {range_max}'
        return result_range
    
    if less is not None and not pd.isna(less):
        result_less = f'less than {less}'
        return result_less

    if more is not None and not pd.isna(more):
        result_more = f'more than {more}'
        return result_more

    if spot is not None:
        result_spot = f'{spot}'
        return result_spot
    
    
    
    
income['standard_income'] = income.apply(lambda row: combined_income(row['range_min_comp'], row['range_max_comp'], row['less_comp'], row['more_comp'], row['spot_comp']), axis=1)

In [57]:
# last result
def result(text_input, standard):
    if isinstance(text_input , str) and standard == None :
        return text_input , 0
    
    elif text_input == standard :
        return text_input , 1
    
    elif text_input != standard :
        return standard , 2
    
    elif text_input == None and standard == None :
        return text_input , 3
    
    
    
income[['std_output' , 'status']] = income.apply(lambda row: pd.Series(result(row['income'], row['standard_income'])), axis=1)

In [58]:
selected_cols = ['id','income','range_min_comp', 'range_max_comp','less_comp','more_comp', 'spot_comp', 'standard_income','validate_length' , 'validate_consecutive','std_output' ,'status']

income = income[selected_cols]

In [59]:
income.head(5)

,id,income,range_min_comp,range_max_comp,less_comp,more_comp,spot_comp,standard_income,validate_length,validate_consecutive,std_output,status
0,income-Range-1,10K - 20K,10000,20000,None,None,None,10000 - 20000,True,False,10000 - 20000,2
1,income-Range-2,15K - 20K,15000,20000,None,None,None,15000 - 20000,True,False,15000 - 20000,2
2,income-Range-3,20K - 35K,20000,35000,None,None,None,20000 - 35000,True,False,20000 - 35000,2
3,income-Range-4,20K - 40K,20000,40000,None,None,None,20000 - 40000,True,False,20000 - 40000,2
4,income-Range-5,40K - 60K,40000,60000,None,None,None,40000 - 60000,True,False,40000 - 60000,2
